# 04. Statistical Analysis
**Objective:** Apply statistical methods — correlation, hypothesis testing, regression — to validate business insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/processed/cleaned_retail_data.csv.gz', compression='gzip')
df_active = df[df['IsCancelled']==False].copy()
print(f"Active orders loaded: {len(df_active):,} rows")

---
## 1. Correlation Analysis
Examine relationships between all key numeric variables.

In [ ]:
num_cols = ['Quantity','UnitPrice','TotalRevenue','AvgOrderValue','Recency','Frequency','Monetary','ItemsPerOrder']
corr = df_active[num_cols].corr().round(2)

plt.figure(figsize=(10,7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1, square=True)
plt.title('Correlation Matrix — Key Numerical Variables')
plt.tight_layout()
plt.show()
print("\nKey insight: High correlation between Frequency-Monetary suggests frequent buyers also spend more.")

---
## 2. Hypothesis Test — Is UK Revenue Significantly Different from International?
**H₀:** Mean revenue per transaction is the same for UK and International customers.
**H₁:** There is a significant difference.

In [ ]:
uk_rev   = df_active[df_active['IsUK']=='UK']['TotalRevenue']
intl_rev = df_active[df_active['IsUK']=='International']['TotalRevenue']

t_stat, p_value = stats.mannwhitneyu(uk_rev, intl_rev, alternative='two-sided')

print(f"Mann-Whitney U Test (non-parametric, suitable for skewed data)")
print(f"  UK mean revenue per transaction    : £{uk_rev.mean():.2f}")
print(f"  International mean per transaction : £{intl_rev.mean():.2f}")
print(f"  U-statistic : {t_stat:.2f}")
print(f"  P-value     : {p_value:.5f}")
if p_value < 0.05:
    print("  RESULT: Reject H₀ — Significant difference exists (p < 0.05)")
else:
    print("  RESULT: Fail to reject H₀ — No significant difference (p >= 0.05)")

---
## 3. Hypothesis Test — Do Weekdays Drive More Revenue Than Weekends?

In [ ]:
weekdays = ['Monday','Tuesday','Wednesday','Thursday','Friday']
weekends = ['Saturday','Sunday']

wd_rev = df_active[df_active['DayOfWeek'].isin(weekdays)]['TotalRevenue']
we_rev = df_active[df_active['DayOfWeek'].isin(weekends)]['TotalRevenue']

t2, p2 = stats.mannwhitneyu(wd_rev, we_rev, alternative='two-sided')
print(f"Mann-Whitney U Test: Weekday vs Weekend Revenue")
print(f"  Weekday mean : £{wd_rev.mean():.2f}")
print(f"  Weekend mean : £{we_rev.mean():.2f}")
print(f"  P-value      : {p2:.5f}")
if p2 < 0.05:
    print("  RESULT: Significant difference — Weekdays and weekends perform differently.")
else:
    print("  RESULT: No significant difference between weekdays and weekends.")

---
## 4. Monthly Revenue — Quarterly Comparison

In [ ]:
quarterly = df_active.groupby('Quarter')['TotalRevenue'].agg(['sum','mean','count']).round(2)
quarterly.columns = ['Total Revenue (£)','Avg Revenue per txn (£)','Transactions']
print(quarterly)

plt.figure(figsize=(8,4))
quarterly['Total Revenue (£)'].plot(kind='bar', color=['#10B981','#2563EB','#F59E0B','#EF4444'], edgecolor='white')
plt.title('Total Revenue by Quarter')
plt.ylabel('Revenue (£)')
plt.xlabel('Quarter')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 5. RFM Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))

rfm_data = df_active[df_active['CustomerID']!='Guest'].drop_duplicates('CustomerID')

axes[0].hist(rfm_data['Recency'].dropna(), bins=40, color='#EF4444', edgecolor='white')
axes[0].set_title('Recency Distribution (days)')
axes[0].set_xlabel('Days since last purchase')

axes[1].hist(rfm_data['Frequency'].dropna().clip(upper=50), bins=40, color='#2563EB', edgecolor='white')
axes[1].set_title('Frequency Distribution')
axes[1].set_xlabel('Number of orders (clipped at 50)')

axes[2].hist(rfm_data['Monetary'].dropna().clip(upper=5000), bins=40, color='#10B981', edgecolor='white')
axes[2].set_title('Monetary Distribution')
axes[2].set_xlabel('Total spend £ (clipped at £5000)')

plt.tight_layout()
plt.show()

---
## 6. Revenue Regression — Does Hour of Day Predict Revenue?

In [ ]:
hourly = df_active.groupby('Hour')['TotalRevenue'].mean().reset_index()
slope, intercept, r_value, p_value, std_err = stats.linregress(hourly['Hour'], hourly['TotalRevenue'])

plt.figure(figsize=(10,4))
plt.scatter(hourly['Hour'], hourly['TotalRevenue'], color='#7C3AED', zorder=5)
plt.plot(hourly['Hour'], slope*hourly['Hour']+intercept, color='#EF4444', linewidth=2, label=f'Regression line (R²={r_value**2:.3f})')
plt.title('Hour of Day vs Average Revenue per Transaction')
plt.xlabel('Hour of Day')
plt.ylabel('Avg Revenue (£)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"R² = {r_value**2:.4f}")
print(f"Slope = {slope:.4f} (revenue change per hour)")
print(f"P-value = {p_value:.5f}")
if p_value < 0.05:
    print("Hour of day is a statistically significant predictor of average revenue.")
else:
    print("Hour of day is NOT a statistically significant predictor of revenue.")